In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install pytorch-msssim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 39.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import functools
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
from typing import List, Tuple, Dict
from pytorch_msssim import MS_SSIM
import math
import time
import h5py
import json
import os
import copy
import random
import ast

# Configuración de dispositivo (GPU si está disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# torch.manual_seed(42)
# random.seed(42)

IMG_DIR = '/content/drive/MyDrive/DPB4 GradCAM/TMAB - Shapley/Pytorch/Data'
MODEL_PATH = '/content/drive/MyDrive/DPB4 GradCAM/TMAB - Shapley/Pytorch/SimpleBeamformer'
CHECKPOINT_FILENAME = 'tmab_shapley_simple_checkpoint.json'

## Models

In [4]:
class SimpleBeamformer(nn.Module):
    def __init__(self, n_channels=2, n_classes=1):
        super(SimpleBeamformer, self).__init__()
        # Encoder
        self.enc1 = nn.Sequential(
            nn.Conv2d(n_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        self.enc2 = nn.Sequential(
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        # Decoder
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        self.dec2 = nn.Sequential(
            nn.Conv2d(32, n_classes, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Encoder
        x1 = self.enc1(x)
        x2 = self.enc2(x1)

        # Decoder
        d1 = self.dec1(x2)
        d2 = self.dec2(d1)# + x1)
        return d2

class SimpleBeamformerV3(nn.Module):
    def __init__(self, n_channels=2, n_classes=1):
        super(SimpleBeamformerV3, self).__init__()

        # Encoder path
        self.enc1 = nn.Sequential(
            nn.Conv2d(n_channels, 32, kernel_size=3, padding=1),
            nn.GELU()
        )

        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=3, stride=2, padding=1),
            nn.GELU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.GELU()
        )

        # Decoder path
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.GELU()
        )

        self.dec2 = nn.Sequential(
            nn.Conv2d(32, n_classes, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(x1)
        d1 = self.dec1(x2)
        d2 = self.dec2(d1)# + x1)
        return d2

## Show Dataset

In [5]:
def process_and_plot(model, input_path, target_path, ax_output, ax_target, device):
    # Cargar y procesar la entrada
    sample = np.load(input_path)
    input_tensor = torch.tensor(sample, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device)

    #Agregamos la dimension latent space z
    # N, _, H, W = input_tensor.size()
    # z = torch.randn(N, 1, H, W).to(device)
    # input_tensor = torch.cat([input_tensor, z], dim=1)
    # print("input shape: ",input_tensor.size())

    # Obtener la salida del modelo
    with torch.no_grad():
        output = model(input_tensor)

    # print("ouput shape: ",output.size())
    # Cargar el target
    target = np.load(target_path)
    # print("target shape: ",target.shape)

    # print('min target', target.min(), '\n')
    # print('max target', target.max(), '\n')
    # print('min output', output.squeeze().min(), '\n')
    # print('max output', output.squeeze().max(), '\n')

    # Configurar las opciones de visualización
    extent_full = [-20, 20, 80, 30]
    opts = {"extent": extent_full, "origin": "upper"}

    # Mostrar la salida del modelo
    ax_output.imshow(output.squeeze().cpu().numpy(), **opts, cmap='gray')
    ax_output.set_title('Output')

    # Mostrar el target
    ax_target.imshow(target, **opts, cmap='gray')
    ax_target.set_title('Target')

# Configurar el dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Cargar el modelo
# model = Wang2020UnetGenerator(input_nc=3, #channel data (2) + latent space z (1)
#                                       output_nc=1,
#                                       num_downs=5,
#                                       ngf=64,
#                                       norm_layer=nn.BatchNorm2d,
#                                       use_dropout=False).to(device)

model = SimpleBeamformer().to(device)
model.load_state_dict(torch.load(f'{MODEL_PATH}/best_model2.pth', map_location=device, weights_only=True))
model.eval()

# Crear el subplot
n_images = 10
fig, axes = plt.subplots(n_images, 2, figsize=(10, 5*n_images))

for i in range(n_images):
    input_path = f'{IMG_DIR}/input_id/simu{i+1:05d}.npy'
    target_path = f'{IMG_DIR}/target_from_raw/simu{i+1:05d}.npy'

    process_and_plot(model, input_path, target_path, axes[i, 0], axes[i, 1], device)
    axes[i, 0].set_ylabel(f'Simu {i+1:05d}')

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [8]:
# model = UNet(n_channels=2, n_classes=1).to(device)

#input = N_batchsxN_channelsxHxW
#input = 1x2x800x128

def process_and_plot(model, input_path, target_path, ax_output, ax_target, device):
    # Cargar y procesar la entrada
    sample = np.load(input_path)
    input_tensor = torch.tensor(sample, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device)

    #Agregamos la dimension latent space z
    # N, _, H, W = input_tensor.size()
    # z = torch.randn(N, 1, H, W).to(device)
    # input_tensor = torch.cat([input_tensor, z], dim=1)
    # print("input shape: ",input_tensor.size())

    # Obtener la salida del modelo
    with torch.no_grad():
        output = model(input_tensor)

    # print("ouput shape: ",output.size())
    # Cargar el target
    target = np.load(target_path)
    # print("target shape: ",target.shape)

    # print('min target', target.min(), '\n')
    # print('max target', target.max(), '\n')
    # print('min output', output.squeeze().min(), '\n')
    # print('max output', output.squeeze().max(), '\n')

    # Configurar las opciones de visualización
    extent_full = [-20, 20, 80, 30]
    opts = {"extent": extent_full, "origin": "upper"}

    # Mostrar la salida del modelo
    ax_output.imshow(output.squeeze().cpu().numpy(), **opts, cmap='gray')
    ax_output.set_title('Output')

    # Mostrar el target
    ax_target.imshow(target, **opts, cmap='gray')
    ax_target.set_title('Target')

# Configurar el dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Cargar el modelo
# model = Wang2020UnetGenerator(input_nc=3, #channel data (2) + latent space z (1)
#                                       output_nc=1,
#                                       num_downs=5,
#                                       ngf=64,
#                                       norm_layer=nn.BatchNorm2d,
#                                       use_dropout=False).to(device)

model = SimpleBeamformerV3().to(device)
# model.load_state_dict(torch.load(f'{MODEL_PATH}/best_model.pth', map_location=device, weights_only=True))
model.load_state_dict(torch.load(f'{MODEL_PATH}/simplev3_mse_weights.pth', map_location=device, weights_only=True))
model.eval()

# Crear el subplot
n_images = 10
fig, axes = plt.subplots(n_images, 2, figsize=(10, 5*n_images))

for i in range(n_images):
    input_path = f'{IMG_DIR}/input_id/simu{i+1:05d}.npy'
    target_path = f'{IMG_DIR}/target_from_raw/simu{i+1:05d}.npy'

    process_and_plot(model, input_path, target_path, axes[i, 0], axes[i, 1], device)
    axes[i, 0].set_ylabel(f'Simu {i+1:05d}')

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.

## Utils

In [9]:
# def calculate_ssim(model: nn.Module, input_path: str, target_path: str) -> float:
#     sample = np.load(input_path)
#     input_tensor = torch.tensor(sample, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device)

#     #Agregamos la dimension latent space z
#     N, _, H, W = input_tensor.size()
#     z = torch.randn(N, 1, H, W).to(device)
#     input_tensor = torch.cat([input_tensor, z], dim=1)

#     with torch.no_grad():
#         output = model(input_tensor)

#     output_np = output.squeeze().cpu().numpy()
#     target_np = np.load(target_path)

#     # Especificamos data_range basado en el rango de los datos
#     data_range = max(np.max(output_np) - np.min(output_np), np.max(target_np) - np.min(target_np))

#     return ssim(output_np, target_np, data_range=1.0)

#-----Multi-Scale Structural Similarity Index
def calculate_ms_ssim(model: nn.Module, input_path: str, target_path: str) -> float:
    sample = np.load(input_path)
    input_tensor = torch.tensor(sample, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device)

    #Agregamos la dimension latent space z
    # N, _, H, W = input_tensor.size()
    # z = torch.randn(N, 1, H, W).to(device)
    # z = torch.load('/content/latent_z.pth').to(device)
    # input_tensor = torch.cat([input_tensor, z], dim=1)

    with torch.no_grad():
        output = model(input_tensor)

    # Cargar el target y convertirlo a tensor
    target_np = np.load(target_path)
    target_tensor = torch.from_numpy(target_np).unsqueeze(dim=0).unsqueeze(dim=0).to(device)
    data_range = max(output.max(), target_tensor.max()) - min(output.min(), target_tensor.min())

    ms_ssim_loss = MS_SSIM(data_range=data_range, size_average=True, channel=1, win_size=7, K = (0.01, 0.03))
    # print('min target', target_tensor.min(), '\n')
    # print('max target', target_tensor.max(), '\n')
    # print('min output', output.min(), '\n')
    # print('max output', output.max(), '\n')

    ms_ssim_value = ms_ssim_loss(output, target_tensor)

    return ms_ssim_value.cpu().item()


#-------MSE
def calculate_mse(model: nn.Module, input_path: str, target_path: str) -> float:
    sample = np.load(input_path)
    input_tensor = torch.tensor(sample, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)

    target_np = np.load(target_path)
    target_tensor = torch.from_numpy(target_np).unsqueeze(dim=0).unsqueeze(dim=0).to(device)

    mse_loss = nn.MSELoss()
    mse_value = mse_loss(output, target_tensor)

    return mse_value.cpu().item()

#------MAPE
def calculate_smape_score(model: nn.Module, input_path: str, target_path: str) -> float:
    """
    Calcula el Symmetric Mean Absolute Percentage Error (sMAPE) y lo convierte a un score.

    Args:
        model: Modelo a evaluar
        input_path: Ruta al archivo de entrada
        target_path: Ruta al archivo objetivo

    Returns:
        float: Score basado en sMAPE, entre 0 y 1 donde 1 es mejor
    """
    sample = np.load(input_path)
    input_tensor = torch.tensor(sample, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)

    target_np = np.load(target_path)
    target_tensor = torch.from_numpy(target_np).unsqueeze(dim=0).unsqueeze(dim=0).to(device)

    # print("Target range:", target_tensor.min().item(), target_tensor.max().item())
    # print("Output range:", output.min().item(), output.max().item())

    # Calcular sMAPE (Symmetric Mean Absolute Percentage Error)
    numerator = torch.abs(target_tensor - output)
    denominator = (torch.abs(target_tensor) + torch.abs(output)) / 2
    smape = torch.mean(numerator / denominator) * 100

    # print("Raw sMAPE:", smape.item())

    # Convertir a score - el sMAPE estará entre 0 y 100
    score = 1 / (1 + smape/100)
    # print("Final score:", score.item())

    return score.item()

#------- GCNR
def calculate_gcnr(im1, im2):
    """
    Calcula el GCNR directamente comparando las distribuciones de intensidad
    de la predicción y el target.

    Args:
        prediction: imagen de predicción
        target: imagen target

    Returns:
        float: valor GCNR
    """
    # Calcular histogramas conjuntos
    _, bins = np.histogram(np.concatenate((im1, im2)), bins=256)

    # Calcular histogramas normalizados
    f, _ = np.histogram(im1, bins=bins, density=True)
    g, _ = np.histogram(im2, bins=bins, density=True)

    # Normalizar los histogramas
    f = f / f.sum()
    g = g / g.sum()

    # Calcular GCNR
    gcnr_value = 1 - np.sum(np.minimum(f, g))

    return gcnr_value

def remove_filters(model: nn.Module, filters: List[Tuple[str, int]]) -> None:
    for layer_name, filter_index in filters:
        parts = layer_name.split('.')
        module = model
        for part in parts[:-1]:
            if part.isdigit():
                module = module[int(part)]
            else:
                module = getattr(module, part)
        layer = getattr(module, parts[-1])
        if isinstance(layer, nn.Conv2d):
            layer.weight.data[int(filter_index)].fill_(0)
            if layer.bias is not None:
                layer.bias.data[int(filter_index)] = 0
        elif isinstance(layer, nn.BatchNorm2d):
            layer.weight.data[int(filter_index)] = 0
            layer.bias.data[int(filter_index)] = 0

def get_filters(model: nn.Module) -> List[Tuple[str, int]]:
    filters = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            filters.extend([(f"{name}.weight", i) for i in range(module.out_channels)])
        elif isinstance(module, nn.BatchNorm2d):
            filters.extend([(f"{name}.weight", i) for i in range(module.num_features)])
    return filters

def process_filter_tuple(filter_info):
    """
    Procesa una tupla de la forma ("('layer_location', filter_number)", shapley_value)
    y retorna (layer_path, filter_number)
    """
    # Extraer y evaluar la tupla interna
    layer_info = ast.literal_eval(filter_info[0])
    layer_path = layer_info[0]
    filter_num = int(layer_info[1])

    # Eliminar las comillas extra si existen
    layer_path = layer_path.strip("'")

    return (layer_path, filter_num)

def navigate_to_layer(model, layer_path):
    """
    Navega a través de la estructura anidada del modelo para encontrar la capa específica
    """
    current_module = model
    parts = layer_path.split('.')

    try:
        for part in parts:
            if part == 'model':
                current_module = current_module.model
            elif part.isdigit():
                current_module = current_module[int(part)]
            elif hasattr(current_module, part):
                current_module = getattr(current_module, part)
            else:
                raise AttributeError(f"No se encontró el atributo '{part}'")

        return current_module
    except Exception as e:
        print(f"Error navegando a {layer_path}: {str(e)}")
        return None

def remove_filters_wang(model, filters):
    """
    Versión actualizada para manejar el formato de filtros
    """
    for filter_info in filters:
        try:
            layer_path, filter_index = process_filter_tuple(filter_info)

            # Encontrar la capa correcta
            parts = layer_path.split('.')
            weight_name = parts[-1]
            parent_path = '.'.join(parts[:-1])

            parent_module = navigate_to_layer(model, parent_path)
            if parent_module is None:
                continue

            # Poner a cero los pesos del filtro
            if isinstance(parent_module, (nn.Conv2d, nn.BatchNorm2d)):
                parent_module.weight.data[filter_index].fill_(0)
                if parent_module.bias is not None:
                    parent_module.bias.data[filter_index] = 0

        except Exception as e:
            print(f"Error al procesar filtro: {str(e)}")
            continue

## Model evaluation

In [ ]:
model = SimpleBeamformerV3().to(device)
model.load_state_dict(torch.load(f'{MODEL_PATH}/simplev3_mse_weights.pth', map_location=device, weights_only=True))
model.eval()

In [ ]:
def analyze_filter_values(filename: str, top_n: int = 50) -> None:
    """
    Analiza los valores de los filtros, mostrando los n valores más altos, más bajos y más cercanos a cero.

    Args:
        filename: Ruta al archivo checkpoint JSON
        top_n: Número de valores top/bottom/zero a mostrar
    """
    # Leer el archivo JSON
    with open(filename, 'r') as f:
        data = json.load(f)

    # Obtener el diccionario de valores y convertirlo a lista de tuplas
    values = data['values']
    value_list: List[Tuple[str, float]] = [(k, float(v)) for k, v in values.items()]

    # Ordenar por valor absoluto para encontrar los más cercanos a cero
    sorted_by_abs = sorted(value_list, key=lambda x: abs(x[1]))

    # Ordenar por valor para encontrar los más altos y más bajos
    sorted_by_value = sorted(value_list, key=lambda x: x[1])

    # Obtener los n valores más bajos, más altos y más cercanos a cero
    lowest_n = sorted_by_value[:top_n]
    highest_n = sorted_by_value[-top_n:][::-1]  # Revertir para mostrar de mayor a menor
    closest_to_zero = sorted_by_abs[:top_n]
    random_filters = random.sample(value_list, top_n)  # Obtener una lista de filtros aleatorios

    # Función auxiliar para imprimir valores
    def print_values(values_list: List[Tuple[str, float]], title: str) -> None:
        print(f"\n{title}")
        print("-" * 80)
        print(f"{'Filtro':<50} | {'Valor':>20}")
        print("-" * 80)

        for filter_name, value in values_list:
            print(f"{filter_name:<50} | {value:>20.10f}")

    # Imprimir resultados
    print_values(highest_n, f"Top {top_n} valores más altos")
    print_values(lowest_n, f"Top {top_n} valores más bajos")
    print_values(closest_to_zero, f"Top {top_n} valores más cercanos a cero")

    # Calcular y mostrar estadísticas generales
    all_values = [v for _, v in value_list]
    print("\nEstadísticas generales")
    print("-" * 80)
    print(f"Valor mínimo: {min(all_values):.10f}")
    print(f"Valor máximo: {max(all_values):.10f}")
    print(f"Promedio: {sum(all_values)/len(all_values):.10f}")
    print(f"Mediana: {np.median(all_values):.10f}")
    print(f"Desviación estándar: {np.std(all_values):.10f}")
    print(f"Total de filtros: {len(all_values)}")

    return highest_n, lowest_n, closest_to_zero, random_filters

checkpoint_path = os.path.join("/content/", "shapley_simplev3_abs_margin_mssim_checkpoint_sample_200.json")
highest_n, lowest_n, closest_to_zero, random_filters = analyze_filter_values(checkpoint_path, top_n=257)

### MS-SSIM vs # filtros removidos

In [ ]:
def plot_triple_comparison(model, base_path, device, highest_n, lowest_n, closest_to_zero, random_filters):
    """
    Grafica el MS-SSIM vs número de filtros removidos para tres grupos de filtros
    """
    def calculate_scores(filters, label):
        scores = []
        model_copy = copy.deepcopy(model)

        # Calcular MS-SSIM inicial
        initial_ssim = calculate_ms_ssim(model_copy,
                                       f'{base_path}/input_id/simu00002.npy',
                                       f'{base_path}/target_from_raw/simu00002.npy')
        scores.append(initial_ssim)
        print(f"\nInitial SSIM for {label}: {initial_ssim:.4f}")

        # Remover filtros uno por uno
        total_filters = len(filters)
        for i in range(total_filters):
            remove_filters_wang(model_copy, [filters[i]])
            ssim = calculate_ms_ssim(model_copy,
                                   f'{base_path}/input_id/simu00002.npy',
                                   f'{base_path}/target_from_raw/simu00002.npy')
            scores.append(ssim)

            # Mostrar progreso cada 10 filtros
            if (i + 1) % 10 == 0:
                print(f"{label}: Processed {i + 1}/{total_filters} filters. Current SSIM: {ssim:.4f}")

        return scores

    # Calcular scores para cada grupo
    print("\nProcessing highest Shapley values...")
    high_scores = calculate_scores(highest_n, "Highest Shapley")

    print("\nProcessing lowest Shapley values...")
    low_scores = calculate_scores(lowest_n, "Lowest Shapley")

    print("\nProcessing closest to zero Shapley values...")
    zero_scores = calculate_scores(closest_to_zero, "Closest to Zero")

    print("\nProcessing random Shapley values...")
    random_scores = calculate_scores(random_filters, "Random")

    # Crear el gráfico
    plt.figure(figsize=(12, 8))
    x = range(len(high_scores))

    # Plotear las tres curvas con diferentes estilos
    plt.plot(x, high_scores, 'b-', label='Highest Shapley', alpha=0.7)
    plt.plot(x, low_scores, 'r-', label='Lowest Shapley', alpha=0.7)
    plt.plot(x, zero_scores, 'g-', label='Closest to Zero', alpha=0.7)
    plt.plot(x, random_scores, 'y--', label='Random', alpha=0.7)

    plt.title('MS-SSIM vs Number of Removed Filters')
    plt.xlabel('Number of Removed Filters')
    plt.ylabel('MS-SSIM')
    plt.grid(True, alpha=0.3)
    plt.legend()

    # Añadir anotaciones para valores iniciales y finales
    plt.annotate(f'Initial: {high_scores[0]:.4f}',
                xy=(0, high_scores[0]),
                xytext=(5, 10),
                textcoords='offset points')

    # Anotaciones para valores finales
    y_offset = [-20, 0, 20, -40]  # Diferentes offsets para evitar superposición
    for scores, label, offset in zip([high_scores, low_scores, zero_scores, random_scores],
                                   ['Highest', 'Lowest', 'Zero', 'Random'],
                                   y_offset):
        plt.annotate(f'{label}: {scores[-1]:.4f}',
                    xy=(len(scores)-1, scores[-1]),
                    xytext=(-60, offset),
                    textcoords='offset points',
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.7))

    plt.tight_layout()
    plt.savefig('ssim_vs_removed_filters_triple.png', dpi=300, bbox_inches='tight')
    plt.show()

    return high_scores, low_scores, zero_scores, random_scores

# Para usar la función:
# highest_n, lowest_n, closest_to_zero = analyze_filter_values(checkpoint_path, top_n=150)
high_scores, low_scores, zero_scores, random_scores = plot_triple_comparison(model,
                                                              IMG_DIR,
                                                              device,
                                                              highest_n,
                                                              lowest_n,
                                                              closest_to_zero,
                                                              random_filters)

### MSE vs # filtros removidos

In [ ]:
def plot_mse_comparison(model, base_path, device, highest_n, lowest_n, closest_to_zero, random_filters):
    """
    Grafica el MSE vs número de filtros removidos para cuatro grupos de filtros:
    - Filtros con valores Shapley más altos
    - Filtros con valores Shapley más bajos
    - Filtros con valores Shapley más cercanos a cero
    - Filtros aleatorios
    """
    def calculate_scores(filters, label):
        scores = []
        model_copy = copy.deepcopy(model)

        # Calcular MSE inicial
        initial_mse = calculate_mse(model_copy,
                                  f'{base_path}/input_id/simu00002.npy',
                                  f'{base_path}/target_from_raw/simu00002.npy')
        scores.append(initial_mse)
        print(f"\nInitial MSE for {label}: {initial_mse:.6f}")

        # Remover filtros uno por uno
        total_filters = len(filters)
        for i in range(total_filters):
            remove_filters_wang(model_copy, [filters[i]])
            mse = calculate_mse(model_copy,
                              f'{base_path}/input_id/simu00002.npy',
                              f'{base_path}/target_from_raw/simu00002.npy')
            scores.append(mse)

            # Mostrar progreso cada 10 filtros
            if (i + 1) % 10 == 0:
                print(f"{label}: Processed {i + 1}/{total_filters} filters. Current MSE: {mse:.6f}")

        return scores

    # Calcular scores para cada grupo
    print("\nProcessing highest Shapley values...")
    high_scores = calculate_scores(highest_n, "Highest Shapley")

    print("\nProcessing lowest Shapley values...")
    low_scores = calculate_scores(lowest_n, "Lowest Shapley")

    print("\nProcessing closest to zero Shapley values...")
    zero_scores = calculate_scores(closest_to_zero, "Closest to Zero")

    print("\nProcessing random values...")
    random_scores = calculate_scores(random_filters, "Random")

    # Crear el gráfico
    plt.figure(figsize=(12, 8))
    x = range(len(high_scores))

    # Plotear las cuatro curvas con diferentes estilos
    plt.plot(x, high_scores, 'b-', label='Highest Shapley', alpha=0.7)
    plt.plot(x, low_scores, 'r-', label='Lowest Shapley', alpha=0.7)
    plt.plot(x, zero_scores, 'g-', label='Closest to Zero', alpha=0.7)
    plt.plot(x, random_scores, 'y--', label='Random', alpha=0.7)

    plt.title('MSE vs Number of Removed Filters')
    plt.xlabel('Number of Removed Filters')
    plt.ylabel('Mean Squared Error')
    plt.grid(True, alpha=0.3)
    plt.legend()

    # Añadir anotaciones para valores iniciales y finales
    plt.annotate(f'Initial: {high_scores[0]:.6f}',
                xy=(0, high_scores[0]),
                xytext=(5, 10),
                textcoords='offset points')

    # Anotaciones para valores finales
    y_offset = [-20, 0, 20, -40]  # Diferentes offsets para evitar superposición
    for scores, label, offset in zip([high_scores, low_scores, zero_scores, random_scores],
                                   ['Highest', 'Lowest', 'Zero', 'Random'],
                                   y_offset):
        plt.annotate(f'{label}: {scores[-1]:.6f}',
                    xy=(len(scores)-1, scores[-1]),
                    xytext=(-60, offset),
                    textcoords='offset points',
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.7))

    plt.yscale('log')  # Usar escala logarítmica para mejor visualización
    plt.tight_layout()
    plt.savefig('mse_vs_removed_filters_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

    return high_scores, low_scores, zero_scores, random_scores

high_scores, low_scores, zero_scores, random_scores = plot_mse_comparison(model,
                                                          IMG_DIR,
                                                          device,
                                                          highest_n,
                                                          lowest_n,
                                                          closest_to_zero,
                                                          random_filters)

### sMAPE vs # filtros removidos

In [ ]:
def plot_smape_triple_comparison(model, base_path, device, highest_n, lowest_n, closest_to_zero, random_filters):
    """
    Grafica el sMAPE score vs número de filtros removidos comparando diferentes grupos de filtros
    """
    def calculate_scores(filters, label):
        scores = []
        model_copy = copy.deepcopy(model)

        # Calcular sMAPE inicial
        initial_smape = calculate_smape_score(model_copy,
                                         f'{base_path}/input_id/simu00002.npy',
                                         f'{base_path}/target_from_raw/simu00002.npy')
        scores.append(initial_smape)
        print(f"\nInitial sMAPE score for {label}: {initial_smape:.4f}")

        # Remover filtros uno por uno
        total_filters = len(filters)
        for i in range(total_filters):
            remove_filters_wang(model_copy, [filters[i]])
            smape = calculate_smape_score(model_copy,
                                     f'{base_path}/input_id/simu00002.npy',
                                     f'{base_path}/target_from_raw/simu00002.npy')
            scores.append(smape)

            # Mostrar progreso cada 10 filtros
            if (i + 1) % 10 == 0:
                print(f"{label}: Processed {i + 1}/{total_filters} filters. Current sMAPE score: {smape:.4f}")

        return scores

    # Calcular scores para cada grupo
    print("\nProcessing highest Shapley values...")
    high_scores = calculate_scores(highest_n, "Highest Shapley")

    print("\nProcessing lowest Shapley values...")
    low_scores = calculate_scores(lowest_n, "Lowest Shapley")

    print("\nProcessing closest to zero Shapley values...")
    zero_scores = calculate_scores(closest_to_zero, "Closest to Zero")

    print("\nProcessing random Shapley values...")
    random_scores = calculate_scores(random_filters, "Random")

    # Crear el gráfico
    plt.figure(figsize=(12, 8))
    x = range(len(high_scores))

    # Plotear las cuatro curvas con diferentes estilos
    plt.plot(x, high_scores, 'b-', label='Highest Shapley', alpha=0.7)
    plt.plot(x, low_scores, 'r-', label='Lowest Shapley', alpha=0.7)
    plt.plot(x, zero_scores, 'g-', label='Closest to Zero', alpha=0.7)
    plt.plot(x, random_scores, 'y--', label='Random', alpha=0.7)

    plt.title('sMAPE Score vs Number of Removed Filters')
    plt.xlabel('Number of Removed Filters')
    plt.ylabel('sMAPE Score')
    plt.grid(True, alpha=0.3)
    plt.legend()

    # Añadir anotaciones para valores iniciales y finales
    plt.annotate(f'Initial: {high_scores[0]:.4f}',
                xy=(0, high_scores[0]),
                xytext=(5, 10),
                textcoords='offset points')

    # Anotaciones para valores finales
    y_offset = [-20, 0, 20, -40]  # Diferentes offsets para evitar superposición
    for scores, label, offset in zip([high_scores, low_scores, zero_scores, random_scores],
                                   ['Highest', 'Lowest', 'Zero', 'Random'],
                                   y_offset):
        plt.annotate(f'{label}: {scores[-1]:.4f}',
                    xy=(len(scores)-1, scores[-1]),
                    xytext=(-60, offset),
                    textcoords='offset points',
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.7))

    plt.tight_layout()
    plt.savefig('smape_vs_removed_filters_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

    return high_scores, low_scores, zero_scores, random_scores

high_scores, low_scores, zero_scores, random_scores = plot_smape_triple_comparison(model,
                                                          IMG_DIR,
                                                          device,
                                                          highest_n,
                                                          lowest_n,
                                                          closest_to_zero,
                                                          random_filters)

### gCNR vs # filtros removidos

In [ ]:
def plot_gcnr_triple_comparison(model, base_path, device, highest_n, lowest_n, closest_to_zero, random_filters,
                            title="GCNR Comparison: Filter Importance Analysis"):
    """
    Grafica el GCNR vs número de filtros removidos comparando filtros importantes,
    menos importantes y más cercanos a cero.
    """
    def calculate_scores(filters, label):
        scores = []
        model_copy = copy.deepcopy(model)

        # Cargar datos
        sample = np.load(f'{base_path}/input_id/simu00002.npy')
        target = np.load(f'{base_path}/target_from_raw/simu00002.npy')

        input_tensor = torch.tensor(sample, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device)
        # N, _, H, W = input_tensor.size()
        # z = torch.load('/content/latent_z.pth').to(device)
        # input_tensor = torch.cat([input_tensor, z], dim=1)

        # Calcular GCNR inicial
        with torch.no_grad():
            output = model_copy(input_tensor)

        # Región de inclusión: 128:228, 39:61
        # Región de fondo: 670:770, 94:116
        inclusion = output[:, :,128:228, 39:61]
        back = output[:, :, 670:770, 94:116]
        initial_gcnr = calculate_gcnr(inclusion.squeeze().cpu().numpy(), back.squeeze().cpu().numpy())
        scores.append(initial_gcnr)

        # Remover filtros uno por uno
        total_filters = len(filters)
        for i in range(total_filters):
            model_copy = copy.deepcopy(model)
            remove_filters_wang(model_copy, filters[:i+1])

            with torch.no_grad():
                output = model_copy(input_tensor)

            inclusion = output[:, :,48:258, 26:75]
            back = output[:, :, 560:770, 64:113]
            gcnr = calculate_gcnr(inclusion.squeeze().cpu().numpy(), back.squeeze().cpu().numpy())
            scores.append(gcnr)

            if (i + 1) % 10 == 0:
                print(f"{label}: Processed {i + 1}/{total_filters} filters. Current GCNR: {gcnr:.4f}")

        return scores

    # Calcular scores para los tres conjuntos de filtros
    print("\nProcessing highest Shapley values...")
    high_scores = calculate_scores(highest_n, "Highest Shapley")

    print("\nProcessing lowest Shapley values...")
    low_scores = calculate_scores(lowest_n, "Lowest Shapley")

    print("\nProcessing closest to zero Shapley values...")
    zero_scores = calculate_scores(closest_to_zero, "Closest to Zero")

    print("\nProcessing random filters...")
    random_scores = calculate_scores(random_filters, "Random")

    # Crear el gráfico
    plt.figure(figsize=(12, 8))
    x = range(len(high_scores))

    plt.plot(x, high_scores, 'b-', label='Highest Shapley', alpha=0.7)
    plt.plot(x, low_scores, 'r-', label='Lowest Shapley', alpha=0.7)
    plt.plot(x, zero_scores, 'g-', label='Closest to Zero', alpha=0.7)
    plt.plot(x, random_scores, 'y--', label='Random', alpha=0.7)

    plt.title(title)
    plt.xlabel('Number of Removed Filters')
    plt.ylabel('GCNR')
    plt.grid(True, alpha=0.3)
    plt.legend()

    # Añadir anotaciones para valores iniciales y finales
    plt.annotate(f'Initial: {high_scores[0]:.4f}',
                xy=(0, high_scores[0]),
                xytext=(5, 10),
                textcoords='offset points')

    # Diferentes offsets para evitar superposición
    y_offset = [-20, 0, 20, 20]
    for scores, label, offset in zip([high_scores, low_scores, zero_scores, random_scores],
                                   ['Highest', 'Lowest', 'Zero', 'Random'],
                                   y_offset):
        plt.annotate(f'{label}: {scores[-1]:.4f}',
                    xy=(len(scores)-1, scores[-1]),
                    xytext=(-60, offset),
                    textcoords='offset points',
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.7))

    plt.tight_layout()
    plt.savefig('gcnr_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

    return high_scores, low_scores, zero_scores, random_scores

# Ejemplo de uso:
high_gcnr, low_gcnr, zero_gcnr, random_gcnr = plot_gcnr_triple_comparison(
    model, IMG_DIR, device, highest_n, lowest_n, closest_to_zero, random_filters,
    "GCNR Analysis: Effect of Filter Removal by Importance"
)

## Degradacion de la imagen

In [ ]:
def compare_filter_removals_single(model, highest_n, title, img_dir, filter_counts=[0, 15, 25, 60, 130, 250], device='cuda'):
    # Crear subplot con una sola fila
    fig, axes = plt.subplots(1, len(filter_counts) + 1, figsize=(20, 4))
    fig.suptitle(f"{title} Shapley values",
                fontsize=16, y=1.02)

    # Set common visualization options
    extent_full = [-20, 20, 80, 30]
    opts = {"extent": extent_full, "origin": "upper", "cmap": "gray"}

    # Load input and target for simu00002
    input_path = f'{img_dir}/input_id/simu00002.npy'
    target_path = f'{img_dir}/target_from_raw/simu00002.npy'

    sample = np.load(input_path)
    target = np.load(target_path)

    # Show target
    axes[0].imshow(target, **opts)
    axes[0].set_title('Target')

    # Process for each filter count
    for i, n_filters in enumerate(filter_counts):
        model_copy = copy.deepcopy(model)

        # Remove filters if needed
        if n_filters > 0:
            remove_filters_wang(model_copy, highest_n[:n_filters])

        # Process input
        input_tensor = torch.tensor(sample, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device)
        # z = torch.load('/content/latent_z.pth').to(device)
        # input_tensor = torch.cat([input_tensor, z], dim=1)

        # Get output
        with torch.no_grad():
            output = model_copy(input_tensor)

        # Plot output
        axes[i+1].imshow(output.squeeze().cpu().numpy(), **opts)
        axes[i+1].set_title(f'{n_filters} filters removed' if n_filters > 0 else 'Original')

    plt.tight_layout()
    return fig

# Ejecutar la comparación
fig = compare_filter_removals_single(model, highest_n, "Highest", IMG_DIR)
plt.show()

fig = compare_filter_removals_single(model, lowest_n, "Lowest", IMG_DIR)
plt.show()

fig = compare_filter_removals_single(model, closest_to_zero, "Closest-to-zero", IMG_DIR)
plt.show()

fig = compare_filter_removals_single(model, random_filters, "Random", IMG_DIR)
plt.show()

## Plotear regiones de interes para metricas de contraste

In [ ]:
def plot_regions_of_interest(model, img_dir, device):
    # Cargar y procesar la imagen
    input_path = f'{img_dir}/input_id/simu00002.npy'
    sample = np.load(input_path)
    input_tensor = torch.tensor(sample, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device)

    # Agregar latent space z
    # z = torch.load('/content/latent_z.pth').to(device)
    # input_tensor = torch.cat([input_tensor, z], dim=1)

    # Obtener la predicción del modelo
    with torch.no_grad():
        output = model(input_tensor)

    # Convertir a numpy para visualización
    output_np = output.squeeze().cpu().numpy()

    # Configurar visualización
    extent_full = [-20, 20, 80, 30]
    opts = {"extent": extent_full, "origin": "upper"}

    plt.figure(figsize=(12, 8))
    plt.imshow(output_np, **opts, cmap='gray')

    # Crear rectángulos en las regiones de interés
    # Región de inclusión: 128:228, 39:61
    inclusion_coord = plt.Rectangle(
        (-20 + 39/128 * 40,  # x start
         30 + 228/800 * 50),  # y start(end)
        22/128 * 40,         # width
        -100/800 * 50,       # height
        fill=False, color='blue', linewidth=2,
        label='Inclusion Region'
    )

    # Región de fondo: 670:770, 94:116
    background_coord = plt.Rectangle(
        (-20 + 94/128 * 40,  # x start
         30 + 770/800 * 50), # y start
        22/128 * 40,         # width
        -100/800 * 50,       # height
        fill=False, color='red', linewidth=2,
        label='Background Region'
    )

    plt.gca().add_patch(inclusion_coord)
    plt.gca().add_patch(background_coord)

    plt.title('Regions of Interest for gCNR Calculation')
    plt.xlabel('Lateral [mm]')
    plt.ylabel('Axial [mm]')
    plt.colorbar(label='Intensity')
    plt.legend()

    plt.show()

# Ejecutar la función
plot_regions_of_interest(model, IMG_DIR, device)

## Perdida de entrenamiento

In [ ]:
# Leer el archivo JSON
with open('/content/history_epoch_1000.json', 'r') as f:
    data = json.load(f)

# Extraer los datos
train_loss = data["train_loss"]
val_loss = data["val_loss"]
epochs_completed = data["epochs_completed"]

# Generar la lista de épocas
epochs = list(range(1, epochs_completed + 1))

# Graficar
plt.figure(figsize=(8, 6))
plt.plot(epochs, train_loss, label='Train Loss', marker='o')
plt.plot(epochs, val_loss, label='Validation Loss', marker='o')

# Añadir etiquetas y título
plt.ylim([0.0115,0.0125])
plt.xlim([400,1000])
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Train Loss vs Validation Loss')
plt.legend()
plt.grid(True)
plt.show()